# EX_05 — Vector stores y retrieval (ejercicios)

**Notebook de referencia:** `notebook/05_Vectorstores_Retrieval.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Chunking

Implementa un chunker trivial por **número de caracteres** con solapamiento (`chunk_size`, `chunk_overlap`). Aplícalo a un texto largo en una lista de strings.


In [1]:
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 40) -> list[str]:
    chunks = []
    
    # Manejo de casos límite: si el texto es más corto que el tamaño del chunk
    if len(text) <= chunk_size:
        return [text]
        
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        
        # El siguiente fragmento retrocede la distancia del 'overlap'
        start += (chunk_size - overlap)
        
        # Romper el bucle si ya alcanzamos el final del texto
        if end >= len(text):
            break
            
    return chunks

# Prueba dummy sugerida en tu notebook:
long_text = "word " * 500  # Genera una cadena larga
chunks_list = chunk_text(long_text)
print(f"Total de chunks generados: {len(chunks_list)}")


Total de chunks generados: 16


## Actividad 2 — Embeddings + FAISS

Embedde los chunks (puede ser `sentence_transformers`) y construye un índice `faiss.IndexFlatIP` o `IndexFlatL2`. Recupera los top-3 para una query.

*Hint:* L2-normalize vectors if you treat inner product as cosine similarity.


In [2]:

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Cargar el modelo de embeddings y vectorizar los chunks
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(chunks_list)  # Devuelve un array de numpy

# Convertir explícitamente a float32 (FAISS solo acepta este tipo de datos)
embeddings = np.array(embeddings).astype("float32")

# 2. Normalizar L2 los vectores (Requisito del Hint para usar IndexFlatIP como similitud coseno)
faiss.normalize_L2(embeddings)

# 3. Construir el índice FAISS
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # IP = Inner Product (Producto Escalar)
index.add(embeddings)                 # Añadir los vectores al índice

# 4. Definir una query de búsqueda y buscar el top-3
query = "What is the context of the word?"
query_embedding = model.encode([query]).astype("float32")
faiss.normalize_L2(query_embedding)   # No olvidar normalizar también la query

k = 3  # Recuperar top-3
scores, indices = index.search(query_embedding, k)

print("Scores (Similitud):", scores)
print("Índices de los chunks recuperados:", indices)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Scores (Similitud): [[0.3264045  0.26892772 0.26892772]]
Índices de los chunks recuperados: [[15  2  1]]


## Actividad 3 — Métrica manual

Para una query y tres documentos **artificiales** (uno relevante, dos ruido), muestra scores de similitud y verifica que el relevante queda primero.


In [3]:
# 1. Definir documentos sintéticos y la query
query_test = "El mecanismo de atención en redes neuronales"
docs = [
    "Ruido absoluto: El cultivo de tomates requiere un riego constante y clima templado.", # Ruido 1
    "Relevante: El mecanismo de atención permite a los Transformers procesar secuencias enfocándose en partes clave.", # RELEVANTE
    "Ruido absoluto: La física cuántica estudia el comportamiento de las partículas subatómicas." # Ruido 2
]

# 2. Obtener los embeddings
q_emb = model.encode([query_test]).astype("float32")
docs_emb = model.encode(docs).astype("float32")

# 3. Normalizar manualmente usando NumPy para el cálculo estricto de similitud coseno
q_emb_norm = q_emb / np.linalg.norm(q_emb, axis=1, keepdims=True)
docs_emb_norm = docs_emb / np.linalg.norm(docs_emb, axis=1, keepdims=True)

# 4. Calcular el score mediante producto escalar (dot product)
# Al estar normalizados, el producto escalar equivale exactamente a la similitud coseno
scores_manuales = np.dot(docs_emb_norm, q_emb_norm.T).flatten()

# 5. Mostrar resultados ordenados de mayor a menor similitud
ranking = sorted(zip(docs, scores_manuales), key=lambda x: x[1], reverse=True)

print("--- RANKING DE DOCUMENTOS RECUPERADOS ---")
for i, (doc, score) in enumerate(ranking, 1):
    print(f"Puesto {i} [Score: {score:.4f}]: {doc}")

--- RANKING DE DOCUMENTOS RECUPERADOS ---
Puesto 1 [Score: 0.4683]: Relevante: El mecanismo de atención permite a los Transformers procesar secuencias enfocándose en partes clave.
Puesto 2 [Score: 0.3844]: Ruido absoluto: La física cuántica estudia el comportamiento de las partículas subatómicas.
Puesto 3 [Score: 0.3777]: Ruido absoluto: El cultivo de tomates requiere un riego constante y clima templado.
